# Regression Task: Minimal Crossing Number of Sankey Layout

In [1]:
import numpy as np
import pandas as pd

import optuna

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from tensorflow.keras.models import load_model

import sys
sys.path.append('..')

from common import data_preprocessing
from common.gcn import generate_cached_data, predict_test_with_best_model_mps

/Users/work/miniconda3/envs/mcbif/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_, y_test_5, _, _, _, = data_preprocessing.prepare_data_vectors(5, 20)
_, y_test_10, _, _, _, = data_preprocessing.prepare_data_vectors(10, 20)

In [3]:
def compute_bootstrapped_r2_ci(y_true, y_pred, n_bootstraps=5000, conf_level=0.95):
    """
    Computes the confidence interval for the R-squared (R2) score using the 
    percentile bootstrap method on a test set.

    Args:
        y_true (array-like): Actual true values for the test set.
        y_pred (array-like): Predicted values for the test set.
        n_bootstraps (int): The number of bootstrap samples to draw. 
                            5000 is generally sufficient for precision.
        conf_level (float): The desired confidence level (e.g., 0.95 for 95% CI).

    Returns:
        tuple: (r2_observed, lower_bound, upper_bound)
               - r2_observed: The R2 score computed on the original full test set.
               - lower_bound: The lower end of the confidence interval.
               - upper_bound: The upper end of the confidence interval.
    """
    
    # 1. Convert inputs to numpy arrays for efficient indexing
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n_samples = len(y_true)

    if n_samples == 0:
        raise ValueError("Input arrays must not be empty.")
        
    # 2. Compute the primary point estimate on the full test set
    r2_observed = r2_score(y_true, y_pred)
    
    bootstrapped_scores = []
    
    # 3. Generate bootstrap samples and compute R2 for each
    for _ in range(n_bootstraps):
        # Draw indices with replacement
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Select data for this bootstrap sample
        true_sample = y_true[indices]
        pred_sample = y_pred[indices]
        
        # Calculate R2 for the sample
        score = r2_score(true_sample, pred_sample)
        bootstrapped_scores.append(score)
        
    # 4. Determine the confidence interval bounds using percentiles
    sorted_scores = np.array(sorted(bootstrapped_scores))
    
    # Calculate the indices for the desired percentiles (e.g., 2.5% and 97.5% for 95% CI)
    alpha = (1.0 - conf_level) / 2
    lower_percentile_index = int(alpha * n_bootstraps)
    upper_percentile_index = int((1 - alpha) * n_bootstraps)

    lower_bound = sorted_scores[lower_percentile_index]
    upper_bound = sorted_scores[upper_percentile_index]
    
    return round(lower_bound,3), round(upper_bound,3)

## N=5

In [4]:
N = 5
M = 20
SEED = 42

### LR

In [5]:
def lr_baseline(N,M):

    (
        y_train_val,
        y_test,
        feature_names,
        feature_names_to_train,
        feature_names_to_test,
    ) = data_preprocessing.prepare_data_vectors(N, M)

    feature_names = [
        "Partitions",
        "HF0",
        "HF1",
        "HF0 & HF1 Stacked",
        "NCE",
        "ARI",
        "MOD",   
    ]

    # Initialize a list to store results
    results = []

    # Test predictions
    y_test_pred_all = {}

    # Iterate over all feature names
    for feature_name in feature_names:
        # Extract training and test data for the current feature
        X_train_val = feature_names_to_train[feature_name]
        X_test = feature_names_to_test[feature_name]

        # Split training data into training and validation sets
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val,
            y_train_val,
            test_size=0.2,
            random_state=SEED,
            stratify=y_train_val,
        )

        # Fit the model on the training data
        lr_model = LinearRegression()
        lr_model.fit(X_train, y_train)

        # Predict on the training and test data
        y_train_pred = lr_model.predict(X_train)
        y_test_pred = lr_model.predict(X_test)
        y_test_pred_all[feature_name] = y_test_pred

        # Evaluate the model on training data
        train_mse = round(mean_squared_error(y_train, y_train_pred), 3)
        train_r2 = round(r2_score(y_train, y_train_pred), 3)

        # Evaluate the model on test data
        test_mse = round(mean_squared_error(y_test, y_test_pred), 3)
        test_r2 = round(r2_score(y_test, y_test_pred), 3)

        # Compute CI for R2 on test data
        lower_ci, upper_ci = compute_bootstrapped_r2_ci(y_test, y_test_pred)

        # Append the results to the list
        results.append({
            "Feature Name": feature_name,
            "Train MSE": train_mse,
            "Train R2": train_r2,
            "Test MSE": test_mse,
            "Test R2": test_r2,
            "Test R2 CI Lower": round(lower_ci, 3),
            "Test R2 CI Upper": round(upper_ci, 3),
        })

    # Convert the results to a dataframe
    return pd.DataFrame(results), y_test_pred_all

In [6]:
lr_results_5, lr_y_test_5 = lr_baseline(5,20)
display(lr_results_5)

,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper
0,Partitions,3.785,0.096,3.858,0.078,0.060,0.095
1,HF0,3.504,0.163,3.568,0.147,0.126,0.167
2,HF1,2.122,0.493,2.150,0.486,0.462,0.509
3,HF0 & HF1 Stacked,1.885,0.550,1.929,0.539,0.517,0.560
4,NCE,2.474,0.409,2.545,0.392,0.366,0.415
5,ARI,3.375,0.194,3.490,0.166,0.142,0.188
6,MOD,2.432,0.419,2.456,0.413,0.388,0.436


### HF1 lower bound

In [7]:
def lower_hf1_bound(N,M):
    (
        y_train_val,
        y_test,
        feature_names,
        feature_names_to_train,
        feature_names_to_test,
    ) = data_preprocessing.prepare_data_images(N, M)

    # Extract training and test data for HF1
    X_train_val = feature_names_to_train["HF1"]
    X_test = feature_names_to_test["HF1"]

    # Split training data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.2,
        random_state=SEED,
        stratify=y_train_val,
    )

    # Compute lower bound for minimal crossing number and take as predictions
    y_train_pred = np.diagonal(X_train,offset=1, axis1=1, axis2=2).sum(axis=1)
    y_test_pred = np.diagonal(X_test,offset=1, axis1=1, axis2=2).sum(axis=1)

    # Evaluate the model on training data
    train_mse = round(mean_squared_error(y_train, y_train_pred), 3)
    train_r2 = round(r2_score(y_train, y_train_pred), 3)

    # Evaluate the model on test data
    test_mse = round(mean_squared_error(y_test, y_test_pred), 3)
    test_r2 = round(r2_score(y_test, y_test_pred), 3)

    # Append the results to the list
    return pd.DataFrame([{
        "Feature Name": "HF1 - lower bound",
        "Train MSE": train_mse,
        "Train R2": train_r2,
        "Test MSE": test_mse,
        "Test R2": test_r2
    }])

In [8]:
lower_hf1_bound(5,20)

,Feature Name,Train MSE,Train R2,Test MSE,Test R2
0,HF1 - lower bound,18.306,-3.372,18.398,-3.397


### CNN

In [9]:
def optuna_results(storage_path, N, model_name="cnn"):
    
    # Initialize a list to store results
    results = []
    y_test_pred_all = {}

    if model_name == "cnn":
        (
            y_train_val,
            y_test,
            feature_names,
            feature_names_to_train,
            feature_names_to_test,
        ) = data_preprocessing.prepare_data_images(N, 20)
    else:
        (
            y_train_val,
            y_test,
            feature_names,
            feature_names_to_train,
            feature_names_to_test,
        ) = data_preprocessing.prepare_data_vectors(N, 20)

    feature_names = [
        "Partitions",
        "HF0",
        "HF1",
        "HF0 & HF1 Stacked",
        "NCE",
        "ARI",
        "MPO",   
    ]

    # Add MPO as a copy of MOD
    feature_names_to_test["MPO"] = feature_names_to_test["MOD"]

    for feature_name in feature_names:
        try:
            study_name =  f"{feature_name}_N{N}_M{M}"
            study = optuna.load_study(study_name=study_name, storage=storage_path)
            df_study = study.trials_dataframe()
            df_study.sort_values("value", inplace=True, ignore_index=True)

            # Prepare result feature dictionary
            result_feature = {
                    "Feature Name": feature_name,
                    "Train MSE": round(df_study["user_attrs_train_loss"][0],3),
                    "Train R2": round(df_study["user_attrs_train_r2_score"][0],3),
                    "Test MSE": round(df_study["user_attrs_test_loss"][0],3),
                    "Test R2": round(df_study["user_attrs_test_r2_score"][0],3),        
                }
           
            # load best model
            model_path = df_study["user_attrs_model_path"][0].replace("cnn/optuna","optuna/cnn")
            model = load_model(model_path)
            X_test = feature_names_to_test[feature_name]

            if len(X_test.shape) == 3:
                X_test = X_test[..., np.newaxis]

            # Compute y_test_pred
            y_test_pred = model.predict(X_test)
            y_test_pred_all[feature_name] = y_test_pred.flatten()

            # Compute CI for R2 on test data
            lower_ci, upper_ci = compute_bootstrapped_r2_ci(y_test, y_test_pred)
            result_feature["Test R2 CI Lower"] = round(lower_ci, 3)
            result_feature["Test R2 CI Upper"] = round(upper_ci, 3)

            if model_name == "cnn":
                result_feature["Learning Rate"] = df_study["params_learning_rate"][0]
                result_feature["Filters"] = df_study["params_filters"][0]
                result_feature["Kernel Size"] = df_study['params_kernel_size'][0]
            elif model_name == "mlp":
                result_feature["Learning Rate"] = df_study["params_learning_rate"][0]
                result_feature["Nodes"] = df_study["params_n_nodes"][0]
                result_feature["Layers"] = df_study['params_n_layers'][0]
                result_feature["Dropout"] = df_study['params_dropout_rate'][0]
            results.append(result_feature)

            
        except:
            continue
    # Convert the results to a dataframe
    return pd.DataFrame(results), y_test_pred_all

In [10]:
cnn_results_5, cnn_y_test_5 = optuna_results("sqlite:///results/optuna/cnn/gridsearch_M20_optuna.db", 5, model_name="cnn")
display(cnn_results_5.iloc[1:])

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 580us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 962us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 638us/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Filters,Kernel Size
1,HF0,3.474,0.170,3.536,0.155,0.133,0.176,0.0001,32,3
2,HF1,2.055,0.509,2.076,0.504,0.481,0.525,0.0010,4,2
3,HF0 & HF1 Stacked,1.834,0.562,1.906,0.544,0.523,0.565,0.0010,4,3
4,NCE,2.029,0.515,2.126,0.492,0.469,0.514,0.0050,8,2


In [11]:
cnn_results_5_ari_mpo, cnn_y_test_5_ari_mpo = optuna_results(f"sqlite:///results/optuna/cnn/251121_gridsearch_M{M}_optuna.db", 5, model_name="cnn")
display(cnn_results_5_ari_mpo)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 696us/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Filters,Kernel Size
0,ARI,2.242,0.464,2.418,0.422,0.397,0.446,0.005,8,2
1,MPO,2.263,0.460,2.702,0.354,0.327,0.379,0.001,16,3


In [12]:
cnn_results_5_partitions, cnn_y_test_5_partitions = optuna_results(f"sqlite:///results/optuna/cnn/251123_partitions_gridsearch_M{M}_optuna.db", 5)
display(cnn_results_5_partitions)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 513us/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Filters,Kernel Size
0,Partitions,2.842,0.321,3.067,0.267,0.241,0.292,0.001,16,4


### MLP

In [13]:
mlp_results_5, mlp_y_test_5 = optuna_results(f"sqlite:///results/optuna/mlp/mlp_gridsearch_M{M}_optuna.db", 5, model_name="mlp")
display(mlp_results_5.iloc[1:])

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 426us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 491us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 437us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 568us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Nodes,Layers,Dropout
1,HF0,3.516,0.160,3.555,0.150,0.131,0.170,0.0005,256,1,0.0
2,HF1,2.099,0.499,2.128,0.491,0.468,0.513,0.0001,8,1,0.0
3,HF0 & HF1 Stacked,1.896,0.547,1.921,0.541,0.520,0.561,0.0005,256,1,0.5
4,NCE,2.349,0.439,2.473,0.409,0.385,0.432,0.0050,4,1,0.0


In [14]:
mlp_results_5_part_ari_mpo, mlp_y_test_5_part_ari_mpo = optuna_results(f"sqlite:///results/optuna/mlp/251121_mlp_gridsearch_M{M}_optuna.db", 5, model_name="mlp")
display(mlp_results_5_part_ari_mpo)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 442us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 481us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 457us/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Nodes,Layers,Dropout
0,Partitions,3.124,0.254,3.751,0.104,0.083,0.124,0.0001,256,1,0.25
1,ARI,2.655,0.366,3.288,0.214,0.190,0.237,0.0001,256,1,0.50
2,MPO,2.528,0.396,2.716,0.351,0.325,0.375,0.0005,32,2,0.25


### GCN

In [15]:
N = 5
M = 20
study_version = "pyg_gcn_cuda_layers"
study_name = f"gcn_sankey_pyg_N{N}_M{M}_{study_version}"
storage_path = f"sqlite:///results/optuna/gcn/251216_gcn_pyg_M{M}.db"
study = optuna.load_study(study_name=study_name, storage=storage_path)
df_study = study.trials_dataframe()
df_study.sort_values("value", inplace=True, ignore_index=True)
df_study.iloc[0]

number                                                                     136
value                                                                 2.483354
datetime_start                                      2025-12-17 19:55:17.026563
datetime_complete                                   2025-12-17 20:44:31.769663
duration                                                0 days 00:49:14.743100
params_dropout                                                             0.0
params_hidden_dim                                                          128
params_lr                                                                 0.01
params_num_layers                                                            3
params_weight_decay                                                        0.0
user_attrs_gpu_id                                                            3
user_attrs_history_path      results/optuna/gcn_cuda_layers/models/history_...
user_attrs_test_loss                                

In [16]:
generate_cached_data(5,20)
gcn_y_test_pred_5, _, _ = predict_test_with_best_model_mps(5,20)

[Cache Generation] Starting for N=5, M=20
[Cache] Loaded Sankey adjacencies+layers from cache/sankey_gcn_v2
[Cache] Loaded PyG graphs (train) from cache/pyg_graphs_v2/pyg_train_N5_M20_f3.pt
[Cache] Loaded PyG graphs (val) from cache/pyg_graphs_v2/pyg_val_N5_M20_f3.pt
[Cache] Loaded PyG graphs (test) from cache/pyg_graphs_v2/pyg_test_N5_M20_f3.pt
[Cache Generation] Complete for N=5, M=20
  Train: 12784 graphs
  Val: 3197 graphs
  Test: 3996 graphs
[Inference/MPS] N=5, M=20, Best trial #136, Test R2=0.4156


In [17]:
compute_bootstrapped_r2_ci(y_test_5, gcn_y_test_pred_5)

(np.float64(0.392), np.float64(0.437))

## N=10

### LR

In [18]:
lr_results_10, lr_y_test_10 = lr_baseline(10,20)
display(lr_results_10)

,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper
0,Partitions,64.925,0.061,66.158,0.038,0.022,0.052
1,HF0,53.912,0.220,54.029,0.214,0.191,0.236
2,HF1,37.653,0.456,37.957,0.448,0.425,0.470
3,HF0 & HF1 Stacked,33.090,0.522,33.258,0.516,0.495,0.537
4,NCE,37.075,0.464,37.347,0.457,0.434,0.478
5,ARI,51.529,0.255,51.863,0.246,0.221,0.270
6,MOD,43.601,0.370,45.029,0.345,0.321,0.369


### HF1 lower bound

In [19]:
lower_hf1_bound(10,20)

,Feature Name,Train MSE,Train R2,Test MSE,Test R2
0,HF1 - lower bound,2177.263,-30.482,2179.605,-30.703


### CNN

In [20]:
cnn_results_10, cnn_y_test_10 = optuna_results("sqlite:///results/optuna/cnn/250910_cnn_gridsearch_M20_optuna.db", 10, model_name="cnn")
display(cnn_results_10)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 944us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Filters,Kernel Size
0,HF0,53.925,0.220,54.240,0.211,0.188,0.233,0.0005,8,3
1,HF1,37.624,0.456,37.963,0.448,0.425,0.470,0.0005,4,2
2,HF0 & HF1 Stacked,33.238,0.519,33.880,0.507,0.486,0.527,0.0050,4,3
3,NCE,36.222,0.476,37.540,0.454,0.431,0.476,0.0050,16,4


In [21]:
cnn_results_10_ari_mpo, cnn_y_test_10_ari_mpo = optuna_results(f"sqlite:///results/optuna/cnn/251121_gridsearch_M{M}_optuna.db", 10, model_name="cnn")
display(cnn_results_10_ari_mpo)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Filters,Kernel Size
0,ARI,43.127,0.376,48.521,0.294,0.268,0.320,0.005,64,2
1,MPO,43.724,0.368,47.335,0.312,0.286,0.336,0.001,16,3


In [22]:
cnn_results_10_partitions, cnn_y_test_10_partitions = optuna_results(f"sqlite:///results/optuna/cnn/251121_partitions_gridsearch_M{M}_optuna.db", 10, model_name="cnn")
display(cnn_results_10_partitions)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Filters,Kernel Size
0,Partitions,61.392,0.112,63.806,0.072,0.05,0.092,0.01,8,3


### MLP

In [23]:
mlp_results_10, mlp_y_test_10 = optuna_results(f"sqlite:///results/optuna/mlp/mlp_gridsearch_M{M}_optuna.db", 10, model_name="mlp")
display(mlp_results_10.iloc[1:])

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 543us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 801us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 551us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Nodes,Layers,Dropout
1,HF0,54.098,0.218,54.152,0.212,0.191,0.233,0.001,128,2,0.00
2,HF1,37.861,0.453,37.817,0.450,0.427,0.472,0.001,32,1,0.25
3,HF0 & HF1 Stacked,33.575,0.515,33.417,0.514,0.493,0.534,0.001,8,2,0.00
4,NCE,36.791,0.468,37.254,0.458,0.435,0.480,0.001,256,1,0.00


In [24]:
mlp_results_10_part_ari_mpo, mlp_y_test_10_part_ari_mpo = optuna_results(f"sqlite:///results/optuna/mlp/251121_mlp_gridsearch_M{M}_optuna.db", 10, model_name="mlp")
display(mlp_results_10_part_ari_mpo)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 632us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 879us/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 519us/step


,Feature Name,Train MSE,Train R2,Test MSE,Test R2,Test R2 CI Lower,Test R2 CI Upper,Learning Rate,Nodes,Layers,Dropout
0,Partitions,61.244,0.114,66.263,0.036,0.019,0.052,0.0001,256,1,0.5
1,ARI,43.715,0.368,51.172,0.256,0.230,0.280,0.0001,256,1,0.0
2,MPO,49.663,0.282,51.856,0.246,0.222,0.268,0.0050,4,2,0.0


### GCN

In [ ]:
N = 10
M = 20
study_version = "pyg_gcn_cuda_layers"
study_name = f"gcn_sankey_pyg_N{N}_M{M}_{study_version}"
storage_path = f"sqlite:///results/optuna/gcn/251216_gcn_pyg_M{M}.db"
study = optuna.load_study(study_name=study_name, storage=storage_path)
df_study = study.trials_dataframe()
df_study.sort_values("value", inplace=True, ignore_index=True)
df_study.iloc[0]

number                                                                     143
value                                                                52.599766
datetime_start                                      2025-12-19 15:35:05.052005
datetime_complete                                   2025-12-19 16:10:33.153139
duration                                                0 days 00:35:28.101134
params_dropout                                                            0.25
params_hidden_dim                                                          128
params_lr                                                                0.005
params_num_layers                                                            3
params_weight_decay                                                        0.0
user_attrs_gpu_id                                                            6
user_attrs_history_path      results/optuna/gcn_cuda_layers/models/history_...
user_attrs_test_loss                                

In [26]:
generate_cached_data(10,20)
gcn_y_test_pred_10, _, _ = predict_test_with_best_model_mps(10,20)

[Cache Generation] Starting for N=10, M=20
[Cache] Loaded Sankey adjacencies+layers from cache/sankey_gcn_v2
[Cache] Loaded PyG graphs (train) from cache/pyg_graphs_v2/pyg_train_N10_M20_f3.pt
[Cache] Loaded PyG graphs (val) from cache/pyg_graphs_v2/pyg_val_N10_M20_f3.pt
[Cache] Loaded PyG graphs (test) from cache/pyg_graphs_v2/pyg_test_N10_M20_f3.pt
[Cache Generation] Complete for N=10, M=20
  Train: 12777 graphs
  Val: 3195 graphs
  Test: 3994 graphs
[Inference/MPS] N=10, M=20, Best trial #143, Test R2=0.2287


In [27]:
compute_bootstrapped_r2_ci(y_test_10, gcn_y_test_pred_10)

(np.float64(0.207), np.float64(0.248))

## Statistical tests

In [28]:
from scipy.stats import wilcoxon

In [29]:
# test if model residuals for best HF0&HF1 model (CNN) are smaller than best CE model (CNN) at N=5
r_hf01_5 = np.abs(cnn_y_test_5['HF0 & HF1 Stacked'] - y_test_5)
r_ce_5 = np.abs(cnn_y_test_5['NCE'] - y_test_5)
w_stat_5, p_value_5 = wilcoxon(r_hf01_5, r_ce_5, alternative='less')
print(f"Wilcoxon signed-rank test (N=5): W-statistic = {w_stat_5}, p-value = {p_value_5}")

Wilcoxon signed-rank test (N=5): W-statistic = 3506886.0, p-value = 1.3217297549252472e-11


In [30]:
# test if model residuals for best HF0&HF1 model (LR) are smaller than best CE model (MLP) at N=10
r_hf01_10 = np.abs(lr_y_test_10['HF0 & HF1 Stacked'] - y_test_10)
r_ce_10 = np.abs(mlp_y_test_10['NCE'] - y_test_10)
w_stat_10, p_value_10 = wilcoxon(r_hf01_10, r_ce_10, alternative='less')
print(f"Wilcoxon signed-rank test (N=10): W-statistic = {w_stat_10}, p-value = {p_value_10}")

Wilcoxon signed-rank test (N=10): W-statistic = 3383660.0, p-value = 4.9412928169288926e-17


In [31]:
# test if model residuals for best HF1 model (LR) are smaller than best GCN model at N=5
r_hf1_5 = np.abs(lr_y_test_5['HF1'] - y_test_5)
r_gcn_5 = np.abs(gcn_y_test_pred_5 - y_test_5)
w_stat_5, p_value_5 = wilcoxon(r_hf1_5, r_gcn_5, alternative='less')
print(f"Wilcoxon signed-rank test (N=10): W-statistic = {w_stat_5}, p-value = {p_value_5}")

Wilcoxon signed-rank test (N=10): W-statistic = 3597566.0, p-value = 2.9491232784273123e-08


In [32]:
# test if model residuals for best HF1 model (LR) are smaller than best GCN model at N=10
r_hf1_10 = np.abs(lr_y_test_10['HF1'] - y_test_10)
r_gcn_10 = np.abs(gcn_y_test_pred_10 - y_test_10)
w_stat_10, p_value_10 = wilcoxon(r_hf1_10, r_gcn_10, alternative='less')
print(f"Wilcoxon signed-rank test (N=10): W-statistic = {w_stat_10}, p-value = {p_value_10}")

Wilcoxon signed-rank test (N=10): W-statistic = 2977556.0, p-value = 4.276654026383921e-44
